# Random Forest: Premier League Match Outcomes
Use the shared 18 pre-match features and the same chronological train/test split as the other model notebooks. Tune the forest using 20 randomized parameter combinations across three expanding training-period folds. Select parameters without using the test season, refit on all training matches, then report test accuracy.


In [10]:
import pandas as pd
import numpy as np
from match_features import MatchFeatureBuilder


In [11]:
# Combine all six seasons into one DataFrame.
csv_files = [
    "20-21.csv",
    "21-22.csv",
    "22-23.csv",
    "23-24.csv",
    "24-25.csv",
    "25-26.csv",
]

# Use the same feature configuration in each model notebook.
feature_builder = MatchFeatureBuilder(windows=(5, 20), min_matches=5)
df = feature_builder.load_csvs(csv_files)

print(f"Loaded {len(df):,} matches and {df.shape[1]} columns.")
df.head()


Loaded 2,280 matches and 24 columns.


,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR
0,E0,12/09/2020,12:30,Fulham,Arsenal,0,3,A,0,1,...,2,6,12,12,2,3,2,2,0,0
1,E0,12/09/2020,15:00,Crystal Palace,Southampton,1,0,H,1,0,...,3,5,14,11,7,3,2,1,0,0
2,E0,12/09/2020,17:30,Liverpool,Leeds,4,3,H,3,2,...,6,3,9,6,9,0,1,0,0,0
3,E0,12/09/2020,20:00,West Ham,Newcastle,0,2,A,0,0,...,3,2,13,7,8,7,2,2,0,0
4,E0,13/09/2020,14:00,West Brom,Leicester,0,3,A,0,0,...,1,7,12,9,2,5,1,1,0,0


### Team form: previous five and up to 20 matches
For both teams, add average points (win = 3, draw = 1, loss = 0), goals scored, and goals conceded over two windows: the previous five matches and up to 20 previous matches. Both home and away games are included, and history carries across seasons.

Both windows require at least five previous recorded Premier League matches. The current match is excluded with `shift(1)`. After calculating the features from the full history, rows missing any of the 12 averages are dropped.

Also add home-minus-away differences for points, goals scored, and goals conceded in both windows: six comparisons derived from the 12 averages. Positive points/scoring differences favor the home team; negative goals-conceded differences favor the home team.

The shared `MatchFeatureBuilder` in `match_features.py` handles this preparation. Future model notebooks should import the same class and use `feature_builder.feature_columns` to select their inputs.


In [12]:
# Build every team's history before splitting into train and test sets.
rows_before = len(df)
df = feature_builder.add_features(df)
feature_columns = feature_builder.feature_columns.copy()

print(f"Added {len(feature_columns)} features.")
print(f"Dropped {rows_before - len(df):,} matches with insufficient history.")
print(f"Remaining: {len(df):,} matches and {df.shape[1]} columns.")
df[["Date", "HomeTeam", "AwayTeam", "FTR"] + feature_columns].head()


Added 18 features.
Dropped 91 matches with insufficient history.
Remaining: 2,189 matches and 42 columns.


,Date,HomeTeam,AwayTeam,FTR,home_points_last5,home_goals_scored_last5,home_goals_conceded_last5,home_points_last20,home_goals_scored_last20,home_goals_conceded_last20,...,away_goals_conceded_last5,away_points_last20,away_goals_scored_last20,away_goals_conceded_last20,points_difference_last5,goals_scored_difference_last5,goals_conceded_difference_last5,points_difference_last20,goals_scored_difference_last20,goals_conceded_difference_last20
0,2020-10-24,Fulham,Crystal Palace,A,0.2,0.8,2.4,0.2,0.8,2.4,...,1.6,1.4,1.2,1.6,-1.2,-0.4,0.8,-1.2,-0.4,0.8
1,2020-10-24,Liverpool,Sheffield United,H,2.0,2.6,2.6,2.0,2.6,2.6,...,1.4,0.2,0.4,1.4,1.8,2.2,1.2,1.8,2.2,1.2
2,2020-10-25,Southampton,Everton,H,1.4,1.6,1.8,1.4,1.6,1.8,...,1.4,2.6,2.8,1.4,-1.2,-1.2,0.4,-1.2,-1.2,0.4
3,2020-10-25,Wolves,Newcastle,D,1.8,1.0,1.4,1.8,1.0,1.4,...,1.8,1.4,1.4,1.8,0.4,-0.4,-0.4,0.4,-0.4,-0.4
4,2020-10-25,Arsenal,Leicester,A,1.8,1.6,1.2,1.8,1.6,1.2,...,1.6,1.8,2.4,1.6,0.0,-0.8,-0.4,0.0,-0.8,-0.4


## Prepare the Data for Machine Learning
Use the 12 averages and six home-minus-away differences from the five-match and 20-match windows as inputs and `FTR` as the label (`H` = home win, `D` = draw, `A` = away win).

Train on seasons 2020-21 through 2024-25 and reserve 2025-26 for testing. Keep matches in time order. For each test match, the averages use results available before kickoff, including earlier matches in the test season. This evaluates predicting matches one at a time, with the fitted model held fixed throughout the test season.


In [13]:
# July 1 separates the 2024-25 and 2025-26 seasons.
test_start = pd.Timestamp("2025-07-01")
train_set = df.loc[df["Date"] < test_start].copy()
test_set = df.loc[df["Date"] >= test_start].copy()

X_train = train_set[feature_columns].copy()
y_train = train_set["FTR"].copy()
X_test = test_set[feature_columns].copy()
y_test = test_set["FTR"].copy()

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)
X_train.head()
print(X_train.columns)


Training features: (1814, 18)
Test features: (375, 18)
Index(['home_points_last5', 'home_goals_scored_last5',
       'home_goals_conceded_last5', 'home_points_last20',
       'home_goals_scored_last20', 'home_goals_conceded_last20',
       'away_points_last5', 'away_goals_scored_last5',
       'away_goals_conceded_last5', 'away_points_last20',
       'away_goals_scored_last20', 'away_goals_conceded_last20',
       'points_difference_last5', 'goals_scored_difference_last5',
       'goals_conceded_difference_last5', 'points_difference_last20',
       'goals_scored_difference_last20', 'goals_conceded_difference_last20'],
      dtype='str')


## Tune a Random Forest Classifier
Search tree counts, depth, minimum leaf sizes, feature sampling, and bootstrapping. Evaluate 20 configurations on three chronological folds (60 validation fits plus a refit on all training data).

The search runs configurations in parallel. Each individual forest uses one worker to avoid nested parallelism. The selected model is assigned to `forest_clf` for all later evaluation cells.


In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

param_distribs = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 5, 8, None],
    "min_samples_leaf": [5, 10, 20, 40],
    "max_features": ["sqrt", 0.5, 1.0],
    "bootstrap": [True, False],
}

time_split = TimeSeriesSplit(n_splits=3)
rnd_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=1),
    param_distributions=param_distribs,
    n_iter=20,
    cv=time_split,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1,
    return_train_score=True,
    error_score="raise",
)
rnd_search.fit(X_train, y_train)

print("Best parameters:", rnd_search.best_params_)
print(f"Best validation accuracy: {rnd_search.best_score_:.2%}")

# Already refitted on all training matches; used by the later evaluation cells.
forest_clf = rnd_search.best_estimator_


Best parameters: {'n_estimators': 300, 'min_samples_leaf': 40, 'max_features': 'sqrt', 'max_depth': 8, 'bootstrap': True}
Best validation accuracy: 54.89%


## Inspect Search Results
Show the selected configuration's fold scores and the top parameter combinations. The original fixed forest averaged 54.82% validation accuracy. The winning search score is used for selection and can be optimistic after comparing many configurations.


In [15]:
# Reuse the search scores instead of repeating the validation fits.
forest_scores = np.array([
    rnd_search.cv_results_[f"split{fold}_test_score"][rnd_search.best_index_]
    for fold in range(time_split.n_splits)
])

print("Scores:", forest_scores)
print("Mean:", forest_scores.mean())
print("Standard deviation:", forest_scores.std())

search_results = pd.DataFrame(rnd_search.cv_results_)
search_results.sort_values("rank_test_score")[[
    "mean_test_score",
    "std_test_score",
    "mean_train_score",
    "params",
]].head(10)


Scores: [0.53200883 0.56953642 0.54525386]
Mean: 0.5489330389992642
Standard deviation: 0.015539891156690848


,mean_test_score,std_test_score,mean_train_score,params
14,0.548933,0.015540,0.543922,"{'n_estimators': 300, 'min_samples_leaf': 40, ..."
6,0.548197,0.015748,0.546224,"{'n_estimators': 200, 'min_samples_leaf': 5, '..."
10,0.548197,0.018933,0.598300,"{'n_estimators': 200, 'min_samples_leaf': 10, ..."
4,0.548197,0.017131,0.558592,"{'n_estimators': 100, 'min_samples_leaf': 20, ..."
15,0.547461,0.012997,0.541231,"{'n_estimators': 500, 'min_samples_leaf': 40, ..."
12,0.547461,0.016222,0.542692,"{'n_estimators': 500, 'min_samples_leaf': 5, '..."
1,0.547461,0.017194,0.570807,"{'n_estimators': 300, 'min_samples_leaf': 20, ..."
19,0.546726,0.019854,0.598420,"{'n_estimators': 500, 'min_samples_leaf': 10, ..."
18,0.543782,0.016255,0.544539,"{'n_estimators': 500, 'min_samples_leaf': 40, ..."
0,0.543782,0.010249,0.541231,"{'n_estimators': 100, 'min_samples_leaf': 40, ..."


## Evaluate the Selected Model on the Test Set
Use the forest selected by chronological validation and compare it with always predicting a home win on the same test matches. Training accuracy is also shown to help identify overfitting. Test results are excluded from parameter selection.


In [16]:
from sklearn.metrics import accuracy_score, classification_report

y_train_pred = forest_clf.predict(X_train)
y_predict_test = forest_clf.predict(X_test)
home_predictions = np.full(len(y_test), "H")

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_predict_test)
home_accuracy = accuracy_score(y_test, home_predictions)

print(f"Training accuracy: {train_accuracy:.2%}")
print(f"Test accuracy: {test_accuracy:.2%}")
print(f"Always-home-win accuracy: {home_accuracy:.2%}")
print(f"Difference from baseline: {100 * (test_accuracy - home_accuracy):+.2f} percentage points")
print()
print(classification_report(
    y_test, y_predict_test,
    labels=["H", "D", "A"],
    target_names=["Home win", "Draw", "Away win"],
    zero_division=0,
))


Training accuracy: 55.95%
Test accuracy: 50.13%
Always-home-win accuracy: 42.40%
Difference from baseline: +7.73 percentage points

              precision    recall  f1-score   support

    Home win       0.53      0.80      0.64       159
        Draw       0.00      0.00      0.00       102
    Away win       0.45      0.54      0.49       114

    accuracy                           0.50       375
   macro avg       0.33      0.44      0.38       375
weighted avg       0.36      0.50      0.42       375



In [17]:
from sklearn.metrics import confusion_matrix

# Rows are actual results; columns are predicted results.
conf_mx = confusion_matrix(y_test, y_predict_test, labels=["H", "D", "A"])
pd.DataFrame(
    conf_mx,
    index=pd.Index(["Home win", "Draw", "Away win"], name="Actual"),
    columns=pd.Index(["Home win", "Draw", "Away win"], name="Predicted"),
)


Predicted,Home win,Draw,Away win
Actual,,,
Home win,127,0,32
Draw,59,0,43
Away win,53,0,61


## Feature Importances
Inspect which inputs the forest used most when splitting its trees. These are model importance scores, rather than evidence of a causal relationship.


In [18]:
feature_importances = forest_clf.feature_importances_
sorted(zip(feature_importances, X_train.columns), reverse=True)


[(np.float64(0.2031367623227077), 'points_difference_last20'),
 (np.float64(0.1465237403239873), 'goals_scored_difference_last20'),
 (np.float64(0.10725409182706641), 'goals_conceded_difference_last20'),
 (np.float64(0.10013222967487324), 'home_points_last20'),
 (np.float64(0.06953925543426501), 'home_goals_scored_last20'),
 (np.float64(0.054765240186346285), 'away_points_last20'),
 (np.float64(0.04507446654727828), 'points_difference_last5'),
 (np.float64(0.0420305458902719), 'away_goals_scored_last20'),
 (np.float64(0.04096733187400103), 'away_goals_conceded_last20'),
 (np.float64(0.037943291578284785), 'goals_conceded_difference_last5'),
 (np.float64(0.030934881174205932), 'home_goals_conceded_last20'),
 (np.float64(0.02348878752263664), 'goals_scored_difference_last5'),
 (np.float64(0.023048317206023673), 'home_points_last5'),
 (np.float64(0.018456803750584613), 'home_goals_scored_last5'),
 (np.float64(0.016394424236384254), 'away_goals_conceded_last5'),
 (np.float64(0.013881394145